In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import numpy as np
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
class CFG:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data_dir = './raw_data/'
    labels_file = 'labels.csv'

In [ ]:
df = pd.read_csv(CFG.labels_file)
counts = df['Label'].value_counts().sort_index()
print("Class distribution:\n", counts)

plt.figure(figsize=(8, 5))
df['Label'].value_counts().sort_index().plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Distribution of Labels in Raw Data')
plt.xlabel('Label')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
labels = pd.read_csv(CFG.labels_file)
labels.head()

In [ ]:
processed_dir = './clean_data_2/'
os.makedirs(processed_dir, exist_ok=True)

for i in range(5):
    os.makedirs(os.path.join(processed_dir, str(i)), exist_ok=True)

In [ ]:
def adaptive_crop_and_resize(img, size=384, threshold=10):
    """Remove black borders and resize/crop to target size."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        img = img[y:y+h, x:x+w]
        
    h, w = img.shape[:2]
    if max(h, w) <= 224:
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_CUBIC)
    else:
        img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA)
        s = (512 - size) // 2
        img = img[s:s+size, s:s+size]
    return img

def apply_clahe_rgb(img, clip=2.0, grid=(8, 8)):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clip, grid).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

def apply_clahe_green(img, clip=2.0, grid=(8, 8)):
    b_c, g_c, r_c = cv2.split(img)
    g_c = cv2.createCLAHE(clip, grid).apply(g_c)
    return cv2.merge([b_c, g_c, r_c])

def graham_preprocess(img, sigma=10):
    """4*img - 4*blur + 128"""
    f = img.astype(np.float32)
    blur = cv2.GaussianBlur(f, (0, 0), sigma)
    return np.clip(cv2.addWeighted(f, 4, blur, -4, 128),
                   0, 255).astype(np.uint8)

def circular_roi_mask(img):
    h, w = img.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    cv2.circle(mask, (w // 2, h // 2), min(h, w) // 2, 255, -1)
    return cv2.bitwise_and(img, img, mask=mask)

def preprocess_image(path, size=384):
    img = cv2.imread(str(path))
    if img is None:
        return None
    img = adaptive_crop_and_resize(img, size=size)
    img = graham_preprocess(img)
    img = apply_clahe_rgb(img)
    img = apply_clahe_green(img)
    img = circular_roi_mask(img)
    return img

In [ ]:
# Visualization of preprocessing steps
sample_path = os.path.join(CFG.data_dir, 'img_0_000101.png')
img = cv2.imread(sample_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

steps = []
steps.append(("Original", img_rgb))

# 1. Adaptive Crop & Resize
img_processed = adaptive_crop_and_resize(img)
steps.append(("Crop & Resize", cv2.cvtColor(img_processed, cv2.COLOR_BGR2RGB)))

# 2. Graham's method
img_graham = graham_preprocess(img_processed)
steps.append(("Graham's Process", cv2.cvtColor(img_graham, cv2.COLOR_BGR2RGB)))

# 3. RGB CLAHE
img_clahe_rgb = apply_clahe_rgb(img_graham)
steps.append(("RGB CLAHE", cv2.cvtColor(img_clahe_rgb, cv2.COLOR_BGR2RGB)))

# 4. Green-channel CLAHE
img_clahe_green = apply_clahe_green(img_clahe_rgb)
steps.append(("Green CLAHE", cv2.cvtColor(img_clahe_green, cv2.COLOR_BGR2RGB)))

# 5. Circular ROI Mask
img_final = circular_roi_mask(img_clahe_green)
steps.append(("Circular ROI Mask", cv2.cvtColor(img_final, cv2.COLOR_BGR2RGB)))

# Plotting in one line
plt.figure(figsize=(24, 6))
for i, (title, step_img) in enumerate(steps):
    plt.subplot(1, len(steps), i + 1)
    plt.imshow(step_img)
    plt.title(title, fontsize=28, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
for fn in tqdm(os.listdir(CFG.data_dir)):
    path = os.path.join(CFG.data_dir, fn)
    label_row = labels[labels['Image'] == fn]
    
    if label_row.empty or label_row.Label.values[0] == 0:
        continue
        
    label = label_row.Label.values[0]
    processed_img = preprocess_image(path)
    
    if processed_img is not None and label != 0:
        cv2.imwrite(os.path.join(processed_dir, str(label), fn), processed_img)

In [ ]:
import random

print("Starting class augmentation to handle imbalance...")

# Get class counts from the processed directory
class_counts = {i: len(os.listdir(os.path.join(processed_dir, str(i)))) for i in range(5)}
max_count = max(class_counts.values())
print(f"Original class counts: {class_counts}. Target count: {max_count}")

# Define the three specific spatial target transformations
# 1 = Horizontal flip, 0 = Vertical flip, -1 = Horizontal + Vertical flip
flip_codes = [1, 0, -1]

for class_idx, count in class_counts.items():
    if count < max_count:
        num_needed = max_count - count
        class_folder = os.path.join(processed_dir, str(class_idx))
        existing_images = os.listdir(class_folder)
        
        print(f"Class {class_idx} needs {num_needed} additional augmented images.")
        
        for i in tqdm(range(num_needed), desc=f"Augmenting Class {class_idx}"):
            # Randomly select a base image from the already processed ones
            base_img_name = random.choice(existing_images)
            base_img_path = os.path.join(class_folder, base_img_name)
            
            img = cv2.imread(base_img_path)
            if img is not None:
                # Randomly pick one of the flip transformations requested
                flip_code = random.choice(flip_codes)
                aug_img = cv2.flip(img, flip_code)
                
                # Save the augmented image with a unique filename
                fname, ext = os.path.splitext(base_img_name)
                aug_fname = f"{fname}_aug_{i}{ext}"
                cv2.imwrite(os.path.join(class_folder, aug_fname), aug_img)

In [ ]:
import matplotlib.pyplot as plt
import os
import pandas as pd
from PIL import Image
import random

# Load labels if not already loaded
df_labels = pd.read_csv('labels.csv')
raw_data_dir = 'raw_data'

# Group by label and pick samples
samples_per_class = 3
num_classes = 5

fig, axes = plt.subplots(samples_per_class, num_classes, figsize=(20, 12))

for label in range(num_classes):
    # Filter images for this class
    class_df = df_labels[df_labels['Label'] == label]
    
    # Select random samples
    sample_imgs = class_df.sample(samples_per_class)['Image'].tolist()
    
    for i, img_name in enumerate(sample_imgs):
        img_path = os.path.join(raw_data_dir, img_name)
        try:
            img = Image.open(img_path)
            ax = axes[i, label]
            ax.imshow(img)
            if i == 0:
                ax.set_title(f"Class {label} (Raw)", fontsize=16)
            ax.axis('off')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")

plt.tight_layout()
plt.show()